# Demo: Integrating the Retriever with an LLM
### Module 5, Topic 5 — RAG from Scratch

**What you'll see in this notebook:**
1. Wire the Topic 3 retriever directly into an LLM call
2. Ask a question the knowledge base can actually answer — end to end, retrieval to answer
3. Ask a question it can't answer — and watch the system say so, instead of guessing
4. Package the whole thing into one `answer_question()` function

This is the moment every earlier topic in this module comes together.


## Step 0 — Install and Set Up Both Clients

We need both providers from earlier in this module: Anthropic for generation, Voyage AI for embeddings.

In [ ]:
!pip install anthropic voyageai --quiet

In [ ]:
import os
import anthropic
import voyageai
import numpy as np

claude = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
vo = voyageai.Client(api_key=os.environ.get("VOYAGE_API_KEY"))

CHAT_MODEL = "claude-3-5-sonnet-20241022"
EMBED_MODEL = "voyage-4"

print("Both clients ready.")

## Step 1 — Rebuild the Knowledge Base From Topic 3

Same chunks, same embedding step — this is exactly Topic 3's Steps 2–3, brought back so this notebook runs standalone.

In [ ]:
chunks = [
    "Naija One Bank — Flexi Save Account Policy (Effective 2026)\n\nThe Flexi Save account is Naija One Bank's flagship savings product for individual customers.",
    "It is designed for customers who want easy access to their funds while still earning competitive interest.",
    "The account has no monthly maintenance fee as long as the minimum balance is maintained.",
    "Interest is calculated daily and credited monthly at a rate of 4.2% per annum.",
    "The minimum opening balance required to activate the account is NGN 5,000.",
    "To continue earning interest, customers must maintain a minimum balance of NGN 1,000 at all times.",
    "Customers are permitted 3 free withdrawals per month. A fee of NGN 500 applies to each withdrawal beyond this limit.",
    "Withdrawals can be made via the mobile app, at any branch, or through an ATM using the Flexi Save debit card.",
    "Accounts that fall below the minimum balance for more than 60 consecutive days will be automatically converted to a Basic Save account, which does not earn interest.",
    "Customers can reactivate Flexi Save status by restoring the minimum balance.",
]

chunk_embeddings = vo.embed(chunks, model=EMBED_MODEL, input_type="document").embeddings
knowledge_base = list(zip(chunks, chunk_embeddings))

print(f"Knowledge base ready: {len(knowledge_base)} chunks.")

## Step 2 — Bring Back the `retrieve()` Function

Identical to Topic 3 — no changes needed.

In [ ]:
def cosine_similarity(vec_a, vec_b):
    vec_a = np.array(vec_a)
    vec_b = np.array(vec_b)
    return np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b))

def retrieve(query, knowledge_base, k=3):
    query_embedding = vo.embed([query], model=EMBED_MODEL, input_type="query").embeddings[0]
    scores = []
    for chunk_text, chunk_vec in knowledge_base:
        score = cosine_similarity(query_embedding, chunk_vec)
        scores.append((chunk_text, score))
    ranked = sorted(scores, key=lambda pair: pair[1], reverse=True)
    return ranked[:k]

print("retrieve() is ready.")

## Step 3 — Build a Prompt From Retrieved Chunks

This new function takes the retriever's output and assembles it into a single prompt, following the three-piece shape from the slides: instruction, chunks, question.

In [ ]:
def build_prompt(question, retrieved_chunks):
    context = "\n\n".join(chunk_text for chunk_text, score in retrieved_chunks)

    prompt = f"""Use only the information in the DOCUMENT below to answer the QUESTION.
If the DOCUMENT does not contain enough information to answer, say so clearly instead of guessing.

DOCUMENT:
{context}

QUESTION:
{question}
"""
    return prompt

print("build_prompt() is ready.")

## Step 4 — A Relevance Threshold

Before calling the LLM at all, we check whether the top retrieved chunk is actually relevant. If its similarity score is too low, there's no point handing it to the model — we already know the knowledge base doesn't cover this question.

In [ ]:
RELEVANCE_THRESHOLD = 0.4

def is_relevant_enough(retrieved_chunks, threshold=RELEVANCE_THRESHOLD):
    if not retrieved_chunks:
        return False
    top_score = retrieved_chunks[0][1]
    return top_score >= threshold

print("is_relevant_enough() is ready.")

## Step 5 — Wire It All Together: `answer_question()`

This is every piece from this topic (and Topics 1–4) combined into one function: retrieve, check relevance, build a prompt if relevant, call Claude, or return a fallback if not.

In [ ]:
def answer_question(question, knowledge_base):
    retrieved_chunks = retrieve(question, knowledge_base, k=3)

    if not is_relevant_enough(retrieved_chunks):
        return "I don't have information about that in the Flexi Save policy document. Please contact Naija One Bank support directly."

    prompt = build_prompt(question, retrieved_chunks)

    response = claude.messages.create(
        model=CHAT_MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.content[0].text

print("answer_question() is ready. The full pipeline is now one function call.")

## Step 6 — Test It: A Question That Fits

Same withdrawal question used throughout this module.

In [ ]:
question_1 = "How much can I take out of my account each month before I get charged?"

answer_1 = answer_question(question_1, knowledge_base)
print(answer_1)

## Step 7 — Check the Result

The answer should mention 3 free withdrawals per month and the NGN 500 fee — pulled straight from the chunk the retriever found, not invented. This is the full RAG loop working end to end: your question went in, and a grounded, accurate answer came out, with zero manual context-pasting required.

## Step 8 — Test It: A Question That Doesn't Fit

This question is completely outside what the Flexi Save policy document covers.

In [ ]:
question_2 = "What's the interest rate on a Naija One Bank car loan?"

answer_2 = answer_question(question_2, knowledge_base)
print(answer_2)

## Step 9 — Check the Result

Instead of inventing a plausible-sounding car loan rate, the system should return the fallback message — because the relevance check caught the low similarity score before the LLM was ever called. Compare this to Topic 1's demo, where the *same kind of question* produced either a refusal or a hallucination from the model directly. Here, the system catches the problem one layer earlier, deliberately, instead of leaving it to chance.

## What's Next

We have a complete, working RAG pipeline. The one thing left to sharpen is exactly how the prompt in `build_prompt()` is written — Topic 6 goes deep into prompt construction for RAG, including how to make the model cite which chunk an answer came from. Topic 7 then covers how to evaluate whether a RAG system like this one is actually performing well.